# Rango de Frecuencias

En este notebook se buscará predecir la variable objetivo realizando el preprocesamiento de los datos agrupando por frecuencias para así reducir la dimensión. Se usará leave one out cross validation para encontrar los parámetros.

## Bibliotecas

In [1]:
library(glmnet)
library(pracma)

Loading required package: Matrix

Loaded glmnet 4.1-2


Attaching package: ‘pracma’


The following objects are masked from ‘package:Matrix’:

    expm, lu, tril, triu




In [2]:
vasijas_X <- read.csv("Vessel_X.txt", header = FALSE)
vasijas_Y <- read.csv("Vessel_Y.txt", header = FALSE)
oxido_sodio_Y <- c(vasijas_Y$V1)

merged_vasijas <- vasijas_X
merged_vasijas$Y <- oxido_sodio_Y

# Dividimos en set de train y set de hold out

In [3]:
set.seed(5)
hold_out_ind <- sample(seq_len(nrow(vasijas_X)), size = 20)

HOLDOUT_X <- vasijas_X[hold_out_ind, ]
HOLDOUT_Y <- oxido_sodio_Y[hold_out_ind]
TRAIN_Y <- oxido_sodio_Y[-hold_out_ind]
TRAIN_X <- vasijas_X[-hold_out_ind, ]

# Funcion para means

In [4]:
# Función que recibe un dataframe y un tamanio de ventana. Devuelve el en cada columna la media de las vecinas
# en ventanas del tamanio recibido como segundos parametro.  

preprocesamiento_agrupar <- function(X, tamanio_ventana) {
  X_means <- data.frame(matrix(NA, nrow = nrow(X), ncol = ceiling(ncol(X) / tamanio_ventana)))
  last_i <- floor(ncol(X) / tamanio_ventana)
  for (i in 1:last_i) {
    j <- i - 1
    X_means[,i] <- rowMeans(X[, (j * tamanio_ventana + 1): (i * tamanio_ventana)])
  }
  50:50
  num_last_cols <- i * tamanio_ventana + ncol(X) %% tamanio_ventana
  num_first_last_cols <- last_i * tamanio_ventana + 1
  if(num_first_last_cols > num_last_cols){
      return(X_means)
  } else if(num_first_last_cols != num_last_cols) {
    X_means[,ceiling(ncol(X) / tamanio_ventana)] <- rowMeans(X[, num_first_last_cols: num_last_cols])
  } else {
    X_means[,ceiling(ncol(X) / tamanio_ventana)] <- X[, num_first_last_cols]
  }
  return (X_means)
}

In [5]:
#corr_matrix <- cor(X_means_train, method = "pearson")
#corr_matrix

# Funcion para eliminar mas correlacionadas

In [6]:
progress <- function(n, min, max){
    return((n - min)/(max-min)*100)
}

In [7]:
# Devuelve k grupos de numeros sin repetir desordenados entre 1 y nrow.
# Precondicion: nrow debe ser divisible por k.
obtener_indices  <- function(nrow, k){
    indices = list()
    desordenados = sample(seq(1, nrow))
    for(j in seq(0, k-1)){
        nuevos_indices = desordenados[(j*nrow/k+1):((j+1)*nrow/k)]
        indices = c(indices, list(nuevos_indices))
    }
    return(indices)
}

In [8]:
set.seed(5)

n = nrow(TRAIN_X)
k = 5
resultados <- data.frame(
  Ventana =integer(), 
  Alpha   =double(),
  Lambda  =double(),
  Error   =double()
)

indices = obtener_indices(n, k)

for(ventana in seq(2, 30, 1)){
    for(alpha in seq(0, 1, 0.1)){
        for(lambda in logspace(-10, -2, 9)){
            err = 0
            for(i in indices){
                X <- preprocesamiento_agrupar(TRAIN_X, ventana)
                
                X_train_scaled = scale(X[-i,])
                Y_train = TRAIN_Y[-i]
                
                X_test_scaled = scale(X[i,], center=attr(X_train_scaled, "scaled:center"),
                              scale=attr(X_train_scaled, "scaled:scale"))

                Y_test = TRAIN_Y[i]

                model <- glmnet(X_train_scaled, Y_train, lambda=lambda, alpha=alpha)
                Y_pred = predict(model, X_test_scaled)
                err = err + sum((Y_test - Y_pred)^2)
            }
            resultados = rbind(resultados, list(ventana, alpha, lambda, err/n))
        }
    }
    message(progress(ventana, 2, 30))
}
colnames(resultados)  <- c("Ventana", "alpha", "lambda", "Error")

0

3.57142857142857

7.14285714285714

10.7142857142857

14.2857142857143

17.8571428571429



[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297
[1] 301
[1] 297


21.4285714285714



[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298
[1] 301
[1] 298


25

28.5714285714286



...

89.2857142857143



[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281
[1] 301
[1] 281


92.8571428571429



[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291
[1] 301
[1] 291


96.4285714285714

100



In [11]:
resultados

,Ventana,alpha,lambda,Error
,<dbl>,<dbl>,<dbl>,<dbl>
1,2,0.0,1e-10,2.9497634
2,2,0.0,1e-09,2.9497625
3,2,0.0,1e-08,2.9497530
4,2,0.0,1e-07,2.9495172
5,2,0.0,1e-06,2.9477918
6,2,0.0,1e-05,2.9328830
7,2,0.0,1e-04,2.7963696
8,2,0.0,1e-03,2.1341172
9,2,0.0,1e-02,1.0429772


In [12]:
which.min(resultados$Error)

[1] 690

In [13]:
resultados[which.min(resultados$Error),]

,Ventana,alpha,lambda,Error
,<dbl>,<dbl>,<dbl>,<dbl>
690,8,1,1e-05,0.6742172


In [ ]:

modelo = glmnet(ventana=8, alpha=1, lambda=1e-05)
